# 04 - Modelling

This notebook trains the main data mining models: clustering, regression, classification, anomaly detection, and PCA. It loads the prepared hourly dataset created in Notebook 03.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PREPARED_PATH = PROJECT_ROOT / "outputs" / "prepared_hourly_energy.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

energy_df = pd.read_csv(PREPARED_PATH, parse_dates=["datetime"])
print("Loaded prepared data:", energy_df.shape)
display(energy_df.head())

## Feature Sets

Two feature sets are used. The behavior feature set supports clustering, classification, and PCA. The forecasting feature set avoids target leakage by using time and lag features for regression.

In [2]:
behavior_features = [
    "global_reactive_power",
    "voltage",
    "global_intensity",
    "sub_metering_1",
    "sub_metering_2",
    "sub_metering_3",
    "sub_metering_total_wh",
    "unmetered_energy_wh",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
]

forecast_features = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1_power",
    "lag_2_power",
    "lag_24_power",
    "rolling_3_power_mean",
    "rolling_24_power_mean",
]

print("Behavior features:", behavior_features)
print("Forecasting features:", forecast_features)

Behavior features: ['global_reactive_power', 'voltage', 'global_intensity', 'sub_metering_1', 'sub_metering_2', 'sub_metering_3', 'sub_metering_total_wh', 'unmetered_energy_wh', 'hour', 'day_of_week', 'month', 'is_weekend']
Forecasting features: ['hour', 'day_of_week', 'month', 'is_weekend', 'lag_1_power', 'lag_2_power', 'lag_24_power', 'rolling_3_power_mean', 'rolling_24_power_mean']


## Clustering: K-Means and DBSCAN

Clustering groups similar hourly consumption periods. K-Means is tested for several cluster counts, while DBSCAN is used to identify dense regions and noise points.

In [3]:
cluster_sample = energy_df.sample(min(20000, len(energy_df)), random_state=42).sort_index()
x_cluster = cluster_sample[behavior_features].replace([np.inf, -np.inf], np.nan).dropna()
scaled_cluster = StandardScaler().fit_transform(x_cluster)

cluster_rows = []
for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(scaled_cluster)
    cluster_rows.append({
        "algorithm": "K-Means",
        "clusters": k,
        "silhouette": silhouette_score(scaled_cluster, labels),
        "noise_points": np.nan,
    })

dbscan = DBSCAN(eps=1.5, min_samples=20)
dbscan_labels = dbscan.fit_predict(scaled_cluster)
dbscan_cluster_count = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
dbscan_score = silhouette_score(scaled_cluster, dbscan_labels) if dbscan_cluster_count > 1 else np.nan
cluster_rows.append({
    "algorithm": "DBSCAN",
    "clusters": dbscan_cluster_count,
    "silhouette": dbscan_score,
    "noise_points": int((dbscan_labels == -1).sum()),
})

clustering_metrics = pd.DataFrame(cluster_rows)
display(clustering_metrics)
clustering_metrics.to_csv(OUTPUT_DIR / "clustering_metrics.csv", index=False)

,algorithm,clusters,silhouette,noise_points
0,K-Means,2,0.260062,NaN
1,K-Means,3,0.203563,NaN
2,K-Means,4,0.216276,NaN
3,K-Means,5,0.226291,NaN
4,DBSCAN,6,0.118303,2334.0


In [4]:
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
clustered_df = cluster_sample.loc[x_cluster.index].copy()
clustered_df["kmeans_cluster"] = kmeans_final.fit_predict(scaled_cluster)
clustered_df["dbscan_cluster"] = dbscan_labels

pca_2d = PCA(n_components=2, random_state=42)
pca_coordinates = pca_2d.fit_transform(scaled_cluster)
clustered_df["pca_1"] = pca_coordinates[:, 0]
clustered_df["pca_2"] = pca_coordinates[:, 1]

cluster_profile = (
    clustered_df.groupby("kmeans_cluster")[["global_active_power", "global_intensity", "sub_metering_total_wh", "unmetered_energy_wh", "hour", "is_weekend"]]
    .mean()
    .round(3)
)
cluster_profile["records"] = clustered_df.groupby("kmeans_cluster").size()
display(cluster_profile)

clustered_df.to_csv(OUTPUT_DIR / "clustered_sample.csv", index=False)
cluster_profile.reset_index().to_csv(OUTPUT_DIR / "cluster_profile.csv", index=False)

,global_active_power,global_intensity,sub_metering_total_wh,unmetered_energy_wh,hour,is_weekend,records
kmeans_cluster,,,,,,,
0,0.698,2.984,279.624,417.986,10.630,0.000,10065
1,0.783,3.351,287.152,495.871,10.204,1.000,3821
2,2.593,10.983,1409.218,1183.356,15.327,0.327,3591


## Sub-metering Breakdown by Cluster

Each K-Means cluster captures a distinct appliance usage pattern. This chart shows the average Wh per hour from each metering source — kitchen (sub_metering_1), laundry (sub_metering_2), water heater/AC (sub_metering_3), and unmetered — making the clusters interpretable in practical terms.

## Weekend vs Weekday Consumption

Comparing the hourly power distribution between weekdays and weekends shows whether household routines shift on rest days. This directly supports the cluster interpretation — Cluster 1 (high weekend share) vs Cluster 0 (purely weekday).

In [ ]:
wk_df = energy_df.copy()
wk_df["day_type"] = wk_df["is_weekend"].map({0: "Weekday", 1: "Weekend"})

fig = px.box(
    wk_df,
    x="hour",
    y="global_active_power",
    color="day_type",
    title="Consumption Distribution by Hour: Weekday vs Weekend",
    labels={
        "global_active_power": "Global active power (kW)",
        "hour": "Hour of day",
        "day_type": "Day type",
    },
    color_discrete_map={"Weekday": "#1f77b4", "Weekend": "#ff7f0e"},
)
fig.show()

In [ ]:
import plotly.express as px

submetering_cols = ["sub_metering_1", "sub_metering_2", "sub_metering_3", "unmetered_energy_wh"]
submetering_breakdown = (
    clustered_df.groupby("kmeans_cluster")[submetering_cols]
    .mean()
    .round(1)
    .reset_index()
)
display(submetering_breakdown)

fig = px.bar(
    submetering_breakdown.melt(
        id_vars="kmeans_cluster",
        value_vars=submetering_cols,
        var_name="source",
        value_name="avg_wh",
    ),
    x="kmeans_cluster",
    y="avg_wh",
    color="source",
    barmode="group",
    title="Average Sub-metering Energy by Cluster (Wh/hour)",
    labels={"avg_wh": "Average energy (Wh)", "kmeans_cluster": "Cluster", "source": "Metering source"},
)
fig.show()

submetering_breakdown.to_csv(OUTPUT_DIR / "cluster_submetering.csv", index=False)

## Cluster Temporal Heatmap

Each cluster has a distinct time signature. This heatmap shows average power by hour of day and day of week for each cluster — revealing *when* each usage pattern dominates.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Derive a plain-language label for each cluster from its profile
def _cluster_label(cluster_id, profile):
    row = profile.loc[cluster_id]
    if row["global_active_power"] == profile["global_active_power"].max():
        return "Peak Usage"
    elif row["is_weekend"] >= 0.5:
        return "Low Usage — Weekend"
    else:
        return "Low Usage — Weekday"

day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
clusters = sorted(clustered_df["kmeans_cluster"].unique())
cluster_labels = {c: _cluster_label(c, cluster_profile) for c in clusters}

fig = make_subplots(
    rows=1,
    cols=len(clusters),
    subplot_titles=[f"Cluster {c} — {cluster_labels[c]}" for c in clusters],
    shared_yaxes=True,
)

for col_idx, cluster_id in enumerate(clusters, start=1):
    subset = clustered_df[clustered_df["kmeans_cluster"] == cluster_id]
    pivot = (
        subset.groupby(["hour", "day_of_week"])["global_active_power"]
        .mean()
        .unstack(level="day_of_week")
        .reindex(index=range(24), columns=range(7), fill_value=0)
    )
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=day_labels,
            y=list(range(24)),
            colorscale="YlOrRd",
            showscale=(col_idx == len(clusters)),
            colorbar={"title": "Avg kW"} if col_idx == len(clusters) else None,
        ),
        row=1,
        col=col_idx,
    )

fig.update_layout(
    title="Cluster Temporal Pattern: Average Power (kW) by Hour and Day of Week",
    height=500,
    yaxis={"autorange": "reversed", "title": "Hour of day"},
)
fig.show()

print("Cluster legend:")
for c, label in cluster_labels.items():
    row = cluster_profile.loc[c]
    print(f"  Cluster {c} — {label}: avg {row['global_active_power']:.2f} kW, {row['is_weekend']*100:.0f}% weekend")

## Regression: Consumption Prediction

Regression predicts hourly global active power using time and lag features. The split is chronological, so the model trains on earlier hours and tests on later hours.

In [5]:
x_reg = energy_df[forecast_features].replace([np.inf, -np.inf], np.nan)
y_reg = energy_df["global_active_power"]
reg_df = pd.concat([x_reg, y_reg], axis=1).dropna()
x_reg = reg_df[forecast_features]
y_reg = reg_df["global_active_power"]

x_train, x_test, y_train, y_test = train_test_split(x_reg, y_reg, test_size=0.2, shuffle=False)

regression_models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge Regression": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Polynomial Ridge": Pipeline([("poly", PolynomialFeatures(degree=2, include_bias=False)), ("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
}

regression_rows = []
for model_name, model in regression_models.items():
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    regression_rows.append({
        "model_name": model_name,
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
        "r2": r2_score(y_test, predictions),
    })

regression_metrics = pd.DataFrame(regression_rows).sort_values("rmse")
display(regression_metrics)
regression_metrics.to_csv(OUTPUT_DIR / "regression_metrics.csv", index=False)

,model_name,mae,rmse,r2
2,Polynomial Ridge,0.370426,0.540849,0.632569
0,Linear Regression,0.377402,0.556446,0.611071
1,Ridge Regression,0.377411,0.556450,0.611066


## Actual vs Predicted: Where Does the Model Struggle?

The best model (Polynomial Ridge) is plotted against the true test-set values over time. Gaps between the lines reveal where the model under- or over-predicts — typically winter evening peaks and unusual weekend patterns.

In [ ]:
import plotly.express as px

best_model = regression_models["Polynomial Ridge"]
pred_values = best_model.predict(x_test)

pred_df = pd.DataFrame({
    "datetime": energy_df.loc[x_test.index, "datetime"].values,
    "actual": y_test.values,
    "predicted": pred_values,
}).sort_values("datetime")

fig = px.line(
    pred_df.melt(id_vars="datetime", value_vars=["actual", "predicted"],
                 var_name="series", value_name="power_kw"),
    x="datetime",
    y="power_kw",
    color="series",
    title="Actual vs Predicted Global Active Power — Polynomial Ridge (Test Set)",
    labels={"power_kw": "Global active power (kW)", "datetime": "Date", "series": ""},
    color_discrete_map={"actual": "#1f77b4", "predicted": "#ff7f0e"},
)
fig.update_traces(opacity=0.8)
fig.show()

pred_df.to_csv(OUTPUT_DIR / "regression_predictions.csv", index=False)
print(f"Saved {len(pred_df):,} test-set predictions to regression_predictions.csv")

## Classification: High vs Normal Consumption

The classification target is `high_consumption`, created from the top quartile of hourly global active power.

In [6]:
x_cls = energy_df[behavior_features].replace([np.inf, -np.inf], np.nan)
y_cls = energy_df["high_consumption"]
cls_df = pd.concat([x_cls, y_cls], axis=1).dropna()
x_cls = cls_df[behavior_features]
y_cls = cls_df["high_consumption"]

x_train, x_test, y_train, y_test = train_test_split(x_cls, y_cls, test_size=0.2, stratify=y_cls, random_state=42)

classification_models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
}

classification_rows = []
report_lines = []
for model_name, model in classification_models.items():
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    classification_rows.append({"model_name": model_name, "accuracy": accuracy_score(y_test, predictions)})
    report_lines.append(model_name + "\n" + classification_report(y_test, predictions, zero_division=0))

classification_metrics = pd.DataFrame(classification_rows).sort_values("accuracy", ascending=False)
display(classification_metrics)
print("\n\n".join(report_lines))

classification_metrics.to_csv(OUTPUT_DIR / "classification_metrics.csv", index=False)
(OUTPUT_DIR / "classification_report.txt").write_text("\n\n".join(report_lines), encoding="utf-8")

,model_name,accuracy
0,Logistic Regression,0.997712
1,Random Forest,0.995709


Logistic Regression
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2622
           1       1.00      1.00      1.00       874

    accuracy                           1.00      3496
   macro avg       1.00      1.00      1.00      3496
weighted avg       1.00      1.00      1.00      3496


Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2622
           1       0.99      0.99      0.99       874

    accuracy                           1.00      3496
   macro avg       0.99      0.99      0.99      3496
weighted avg       1.00      1.00      1.00      3496



688

## Feature Importance: What Drives High Consumption?

The Random Forest classifier assigns an importance score to each feature based on how much it reduces impurity across all trees. Higher score = stronger predictor of high consumption hours.

In [ ]:
rf_model = classification_models["Random Forest"]
feature_importance_df = (
    pd.DataFrame({"feature": behavior_features, "importance": rf_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
display(feature_importance_df)
feature_importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

## PCA: Feature Space Summary

PCA summarizes how much variance is captured by the first three principal components, providing a complementary view of the feature space alongside the clustering results.

In [ ]:
scaled_behavior = StandardScaler().fit_transform(energy_df[behavior_features].dropna())
pca = PCA(n_components=3, random_state=42)
pca.fit(scaled_behavior)
pca_summary = pd.DataFrame({
    "component": ["PC1", "PC2", "PC3"],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca.explained_variance_ratio_),
})
display(pca_summary)
pca_summary.to_csv(OUTPUT_DIR / "pca_summary.csv", index=False)